# Senior-Level Computer Vision Implementation
### Real-time Face and Eye Detection with OpenCV

This notebook demonstrates a modularized approach to real-time object detection using Haar Cascades. 
Improvements include:
- **Modularity**: Logic encapsulated in reusable functions.
- **Configuration**: Centralized constants for easy tuning.
- **Efficiency**: Removed redundant operations and optimized ROI processing.
- **Robustness**: Added checks for stream status and proper resource cleanup.

In [7]:
import cv2
import numpy as np
from typing import Tuple, Optional

# --- Configuration ---
CASCADE_PATHS = {
    "face": cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml',
    "eye": cv2.data.haarcascades + 'haarcascade_eye.xml'
}

DETECTION_PARAMS = {
    "scaleFactor": 1.1,
    "minNeighbors": 5
}

VISUAL_SETTINGS = {
    "color": (0, 0, 255),  # BGR Red
    "thickness": 2,
    "font_scale": 0.5
}

# --- Initialization ---
def load_cascades(paths: dict) -> dict:
    """Loads Haar Cascade classifiers from provided paths."""
    cascades = {}
    for key, path in paths.items():
        classifier = cv2.CascadeClassifier(path)
        if classifier.empty():
            print(f"Warning: Could not load cascade for {key} at {path}")
        cascades[key] = classifier
    return cascades

classifiers = load_cascades(CASCADE_PATHS)

In [8]:
def detect_and_draw(
    frame: np.ndarray, 
    classifier: cv2.CascadeClassifier, 
    label: str, 
    save_filename: Optional[str] = None,
    verbose: bool = True
) -> np.ndarray:
    """
    Generic detection routine: detects objects, draws bounding boxes, 
    and optionally saves the last detected Region of Interest (ROI).
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    detections = classifier.detectMultiScale(gray, **DETECTION_PARAMS)

    for (x, y, w, h) in detections:
        # Log coordinates to stdout as per original implementation
        if verbose:
            print(f"{x} {y} {w} {h}")

        # Draw bounding box
        cv2.rectangle(
            frame, 
            (x, y), 
            (x + w, y + h), 
            VISUAL_SETTINGS["color"], 
            VISUAL_SETTINGS["thickness"]
        )
        
        # Add label
        cv2.putText(
            frame, label, (x, y - 10), 
            cv2.FONT_HERSHEY_SIMPLEX, 
            VISUAL_SETTINGS["font_scale"], 
            VISUAL_SETTINGS["color"], 
            1
        )

        # Save ROI if requested
        if save_filename:
            roi = frame[y:y+h, x:x+w]
            cv2.imwrite(save_filename, roi)

    return frame

In [9]:
def run_detector():
    """Main entry point for the video processing stream."""
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Camera stream not accessible.")
        return

    print("Stream started. Press 'q' to exit.")
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Perform detections with coordinate logging
            if not classifiers['face'].empty():
                frame = detect_and_draw(frame, classifiers['face'], "Face", "detected_face.png")
            
            if not classifiers['eye'].empty():
                frame = detect_and_draw(frame, classifiers['eye'], "Eye", "detected_eye.png")

            # Display output
            cv2.imshow('AI Vision Stream', frame)

            # Exit condition
            if cv2.waitKey(20) & 0xFF == ord('q'):
                break
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        # Clean shutdown
        cap.release()
        cv2.destroyAllWindows()
        print("Resources released.")

if __name__ == "__main__":
    run_detector()

Stream started. Press 'q' to exit.
304 234 120 120
276 281 49 49
276 284 43 43
275 277 55 55
282 248 89 89
278 285 44 44
278 290 37 37
275 278 53 53
238 212 214 214
269 267 69 69
238 213 221 221
241 216 219 219
304 246 112 112
283 296 30 30
283 279 57 57
241 214 227 227
271 262 80 80
311 247 108 108
284 295 32 32
242 217 220 220
280 288 44 44
314 251 102 102
241 215 222 222
281 296 30 30
241 217 219 219
317 255 89 89
279 277 56 56
247 214 212 212
325 253 81 81
280 261 68 68
249 211 209 209
273 248 78 78
322 238 101 101
248 214 206 206
328 262 62 62
243 208 201 201
258 206 166 166
306 223 85 85
274 230 66 66
232 98 216 216
313 133 90 90
263 153 63 63
228 54 227 227
260 123 54 54
308 105 76 76
227 53 244 244
269 135 46 46
301 94 114 114
224 66 244 244
270 141 49 49
370 138 57 57
303 99 126 126
228 84 248 248
268 152 54 54
377 153 52 52
216 90 264 264
264 168 57 57
371 169 54 54
302 138 123 123
227 113 248 248
263 182 55 55
366 180 57 57
221 118 258 258
263 195 55 55
365 192 57 57
221 139